# YOLO Data Preparation Pipeline

This notebook prepares the brain tumor dataset for YOLO object detection training. It organizes the nested directory structure (where each tumor class contains separate `images/` and `labels/` folders) into a flat YOLO-compatible structure with separate train and validation directories.

## Overview
- **Purpose**: Convert nested dataset structure to YOLO format
- **Input**: Raw data from `../data/raw/Data_Brain/` containing Train and Val subsets
- **Output**: Organized YOLO dataset in `../data/data_yolo/` with proper train/val splits
- **Processing**: Validates image-label pairs and copies only matched files

In [ ]:
# === Cell 1: Imports ===
from pathlib import Path
import shutil
from tqdm import tqdm

# === Cell 2: Define paths ===
raw_data = Path("../data/raw/Data_Brain")  # contains Train and Val
output_data = Path("../data/data_yolo")
(output_data / "images/train").mkdir(parents=True, exist_ok=True)
(output_data / "labels/train").mkdir(parents=True, exist_ok=True)
(output_data / "images/val").mkdir(parents=True, exist_ok=True)
(output_data / "labels/val").mkdir(parents=True, exist_ok=True)


ModuleNotFoundError: No module named 'tqdm'

## Cell 1: Import Required Libraries

This cell imports the necessary Python libraries:

- **`pathlib.Path`**: Object-oriented file path handling for cross-platform compatibility
- **`shutil`**: File operations including copying files while preserving metadata
- **`tqdm`**: Progress bar library to track processing of large file batches

These libraries enable efficient file system operations and user feedback during data processing.

In [ ]:
# === Cell 3: Helper function to process one subset ===
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

def prepare_subset(subset):
    src_folder = raw_data / subset
    print(f"\nProcessing subset: {subset}")
    copied, skipped = 0, 0

    for class_dir in sorted(src_folder.iterdir()):
        if not class_dir.is_dir():
            continue
        img_dir = class_dir / "images"
        lbl_dir = class_dir / "labels"
        if not img_dir.exists() or not lbl_dir.exists():
            print(f" Missing folder in {class_dir}")
            continue

        for img_path in tqdm(list(img_dir.iterdir()), desc=f"{class_dir.name}"):
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            label_path = lbl_dir / f"{img_path.stem}.txt"
            if label_path.exists():
                shutil.copy2(img_path, output_data / "images" / subset.lower() / img_path.name)
                shutil.copy2(label_path, output_data / "labels" / subset.lower() / label_path.name)
                copied += 1
            else:
                skipped += 1
    print(f" Copied {copied} valid pairs,  Skipped {skipped} missing labels.")


## Cell 3: Define Helper Function for Data Processing

This cell defines a reusable function `prepare_subset()` that handles the data preparation logic:

**Function Purpose:**
Processes one data subset (Train or Val) by validating image-label pairs and copying valid files to the output directory.

**Key Components:**

1. **Image Extensions Definition** (`IMG_EXTS`): 
   - Supported formats: `.jpg`, `.jpeg`, `.png`, `.bmp`, `.tiff`
   - Used for filtering non-image files

2. **Processing Logic** (`prepare_subset` function):
   - **Input**: Subset name ("Train" or "Val")
   - **Iteration**: Loops through each tumor class directory within the subset
   - **Validation**: For each image, checks if a corresponding `.txt` label file exists
   - **Copy Operation**: Uses `shutil.copy2()` to preserve file metadata (timestamps, permissions)
   - **Counters**: Tracks copied valid pairs vs. skipped files with missing labels
   - **Progress Display**: Uses tqdm progress bars for each class folder

**Output**: Prints summary statistics showing how many files were successfully copied and how many were skipped due to missing labels.

## Cell 2: Define Directory Paths and Create Output Structure

This cell establishes the file paths and initializes the output directory tree:

**Input Path:**
- `../data/raw/Data_Brain/`: Source directory containing raw dataset with Train and Val subsets
- Each tumor class has nested structure: `ClassName/images/` and `ClassName/labels/`

**Output Path:**
- `../data/data_yolo/`: Root directory for YOLO-formatted dataset
- Creates 4 subdirectories:
  - `images/train/`: Training images
  - `images/val/`: Validation images
  - `labels/train/`: Training annotation files (.txt)
  - `labels/val/`: Validation annotation files (.txt)

The `mkdir(parents=True, exist_ok=True)` ensures all parent directories are created and doesn't error if directories already exist.

In [ ]:
# === Cell 4: Run preparation ===
for subset in ["Train", "Val"]:
    prepare_subset(subset)



Processing subset: Train


Pituitary: 100%|██████████| 1424/1424 [00:11<00:00, 120.38it/s]


 Copied 4737 valid pairs,  Skipped 0 missing labels.

Processing subset: Val


Pituitary: 100%|██████████| 136/136 [00:01<00:00, 130.88it/s]

 Copied 510 valid pairs,  Skipped 2 missing labels.


## Cell 4: Execute Data Preparation Pipeline

This cell executes the data preparation function for both dataset subsets:

**Execution Flow:**
1. Calls `prepare_subset("Train")` to process all training data
   - Validates image-label pairs in the Train subset
   - Copies valid pairs to `../data/data_yolo/images/train/` and `../data/data_yolo/labels/train/`
   
2. Calls `prepare_subset("Val")` to process all validation data
   - Validates image-label pairs in the Val subset
   - Copies valid pairs to `../data/data_yolo/images/val/` and `../data/data_yolo/labels/val/`

**Expected Output:**
- Progress bars showing processing status for each tumor class
- Summary statistics for each subset (number of copied files and skipped files)
- Final YOLO-formatted dataset ready for training